# SimCLR — A Simple Framework for Contrastive Learning of Visual Representations (2020)

_Papers · Self-supervised & Representation_

**Pull two augmented views of the same image together and push every other image apart — no labels needed.**

---

This notebook is a practice scaffold for the **SimCLR — A Simple Framework for Contrastive Learning of Visual Representations (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: NT-Xent loss for one positive pair (i=1, j=2). ---
# Tiny batch N=2 -> 2N=4 unit vectors; cosine sim = dot product; tau = 0.5.
zc = torch.tensor([[1.0, 0.0],   # z1  anchor i
                   [0.8, 0.6],   # z2  positive partner j
                   [-0.6, 0.8],  # z3  negative
                   [0.0, -1.0]]) # z4  negative
tau = 0.5
sims = zc[0] @ zc.t()                      # sim(z1, z_k) for all k
exps = torch.exp(sims / tau)
denom = exps[1:].sum()                     # k != 1  (drop the self term)
p_12  = exps[1] / denom                    # softmax prob of the true partner
loss_12 = -torch.log(p_12)
print("worked example:  p_12 =", round(p_12.item(), 4), " loss_12 =", round(loss_12.item(), 4))
# worked example:  p_12 = 0.7919  loss_12 = 0.2333


# --- 1. Encoder f and projection head g (built by hand from nn primitives). ---
class Encoder(nn.Module):                  # small conv net -> representation h
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 7x7
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)
    def forward(self, x): return F.relu(self.fc(self.net(x)))            # h

class ProjHead(nn.Module):                 # MLP, one hidden layer + ReLU:  z = W2 sigma(W1 h)
    def __init__(self, fin=64, hid=64, out=32):
        super().__init__(); self.l1 = nn.Linear(fin, hid); self.l2 = nn.Linear(hid, out)
    def forward(self, h): return self.l2(F.relu(self.l1(h)))            # z


# --- 2. The NT-Xent loss (Eqn. 1), built by hand. z is the 2N stacked projections [v1 ; v2]. ---
def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1)              # cosine similarity needs unit vectors
    N = z.shape[0] // 2
    sim = z @ z.t() / tau                  # 2N x 2N scaled cosine-similarity matrix
    sim.fill_diagonal_(-1e9)               # the indicator 1[k != i]: drop the self term
    # row i's positive is its partner view: rows 0..N-1 pair with N..2N-1 and vice versa.
    targets = torch.cat([torch.arange(N) + N, torch.arange(N)]).to(z.device)
    return F.cross_entropy(sim, targets)   # softmax over similarities = NT-Xent


# --- 3. Two-view augmentation + an UNLABELLED MNIST subset (torchvision, preinstalled). ---
aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])     # used ONLY for the probe later

# --- 4. Pretrain SimCLR (no labels). ---
enc, proj = Encoder().to(device), ProjHead().to(device)
opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()), lr=1e-3)
enc.train(); proj.train(); B = 128
for ep in range(15):
    perm = np.random.permutation(len(imgs)); tot = 0.0; nb = 0
    for s in range(0, len(imgs), B):
        bi = perm[s:s + B]
        v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
        v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
        z  = proj(enc(torch.cat([v1, v2])))         # 2N projections
        loss = nt_xent(z)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
    if ep % 3 == 0: print(f"  pretrain ep {ep}  NT-Xent loss {tot/nb:.3f}")

# --- 5. FREEZE the encoder, extract features h (linear-evaluation protocol, paper S2.3). ---
enc.eval()
with torch.no_grad():
    feats = enc(torch.stack([base(im) for im in imgs]).to(device)).cpu()

def linear_probe(n_lab):                    # train ONLY a linear classifier on frozen h
    accs = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]; te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10); o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad(); F.cross_entropy(clf(feats[tr]), labels[tr]).backward(); o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

def from_scratch(n_lab):                     # train a fresh conv net end-to-end on the few labels
    accs = []
    for seed in range(3):
        torch.manual_seed(seed); g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]; te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64, 10)); o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60):
            o.zero_grad(); F.cross_entropy(net(Xtr), labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

print("\nlabels | probe(frozen SimCLR) | from-scratch")
for n in [20, 50, 100, 300]:
    print(f"{n:6d} |        {linear_probe(n):.3f}         |    {from_scratch(n):.3f}")
# The frozen-SimCLR linear probe beats from-scratch at every label budget -- biggest gap when labels are fewest.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_In the low-label regime, does a linear probe on frozen SimCLR features beat a from-scratch classifier?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)

# Pretrain a tiny SimCLR encoder on UNLABELLED MNIST, freeze it, then compare a linear
# probe on its frozen features vs a from-scratch net -- across label budgets (toy reproduction).
class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Linear(32, feat)
    def forward(s, x): return F.relu(s.fc(s.net(x)))
class ProjHead(nn.Module):
    def __init__(s, fin=64, hid=64, out=32):
        super().__init__(); s.l1=nn.Linear(fin,hid); s.l2=nn.Linear(hid,out)
    def forward(s, h): return s.l2(F.relu(s.l1(h)))

def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1); N = z.shape[0]//2
    sim = z @ z.t() / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(N)+N, torch.arange(N)])
    return F.cross_entropy(sim, tgt)

aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5,1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]; labels = torch.tensor([raw[i][1] for i in idx])

enc, proj = Encoder(), ProjHead()
opt = torch.optim.Adam(list(enc.parameters())+list(proj.parameters()), lr=1e-3)
enc.train(); proj.train(); B=128
for ep in range(15):
    perm = np.random.permutation(len(imgs))
    for s0 in range(0, len(imgs), B):
        bi = perm[s0:s0+B]
        v1 = torch.stack([aug(imgs[i]) for i in bi]); v2 = torch.stack([aug(imgs[i]) for i in bi])
        loss = nt_xent(proj(enc(torch.cat([v1, v2]))))
        opt.zero_grad(); loss.backward(); opt.step()

enc.eval()
with torch.no_grad():
    feats = enc(torch.stack([base(im) for im in imgs]))

def probe(n):
    a=[]
    for seed in range(3):
        g=np.random.RandomState(seed); tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        clf=nn.Linear(feats.shape[1],10); o=torch.optim.Adam(clf.parameters(),lr=0.05)
        for _ in range(200): o.zero_grad(); F.cross_entropy(clf(feats[tr]),labels[tr]).backward(); o.step()
        with torch.no_grad(): a.append((clf(feats[te]).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)
def scratch(n):
    a=[]
    for seed in range(3):
        torch.manual_seed(seed); g=np.random.RandomState(seed)
        tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        net=nn.Sequential(Encoder(), nn.Linear(64,10)); o=torch.optim.Adam(net.parameters(),lr=1e-3)
        Xtr=torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60): o.zero_grad(); F.cross_entropy(net(Xtr),labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte=torch.stack([base(imgs[i]) for i in te]); a.append((net(Xte).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)

for n in [20,50,100,300]:
    print(n, "probe", probe(n), "scratch", scratch(n))
# probe (frozen SimCLR) > from-scratch at every budget; biggest gap at the fewest labels.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline. You pretrained an encoder with SimCLR on unlabelled images, froze it, and
            trained a linear probe on just 20 labels; you also trained a from-scratch model on the same 20
            labels. The probe scores much higher. What does that demonstrate, and what is the one-line ablation
            that would break the SimCLR advantage?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the two accuracies at the smallest label budget; the frozen-SimCLR probe wins. — _With only 20 labels a from-scratch net has too little signal to learn good features; the probe inherits features already learned from thousands of unlabelled images._
- Ablate: replace the trained encoder with a random (untrained) encoder and probe that. — _If the advantage came from the architecture or the probe, a random encoder would do as well. It does not — isolating the pretraining as the cause._
- Conclude that NT-Xent pretraining, not labels or architecture, supplied the useful structure. — _Same encoder, same probe, same labels; only the pretraining differs._

**Answer:** It demonstrates that unlabelled contrastive pretraining transfers: in the low-label
                 regime, a linear probe on frozen SimCLR features beats a from-scratch model because the features
                 were already shaped by thousands of unlabelled images. Probing a random untrained encoder
                 is the ablation that breaks it — the probe accuracy collapses, isolating the NT-Xent
                 pretraining (not the architecture or the probe) as the source of the gain. Our CODEVIZ panel
                 shows the probe beating from-scratch across the label budgets.

</details>

**Problem 2.** Your worked example gave $\ell_{1,2} = 0.233$ with the partner most similar. Now suppose a negative
            view became more similar to the anchor than the true partner — say $\mathrm{sim}(z_1,z_3)$
            jumps from $-0.6$ to $0.9$ while $\mathrm{sim}(z_1,z_2)$ stays $0.8$ (keep $\tau=0.5$). Does the loss
            go up or down, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute the changed term: $e^{0.9/0.5}=e^{1.8}=6.05$ replaces $e^{-1.2}=0.30$ in the denominator. — _A more-similar negative makes the denominator larger._
- New denominator $= 4.953 + 6.05 + 1.0 = 12.0$; numerator unchanged at $4.953$. — _The numerator only depends on the positive pair, which did not change._
- New probability $p_{1,2} = 4.953/12.0 = 0.413$, so $\ell_{1,2} = -\log(0.413) = 0.88$. — _A bigger denominator shrinks the softmax probability of the true partner, raising $-\log p$._

**Answer:** The loss goes up, from $0.233$ to about $0.88$. A negative that is more similar
                 than the true partner inflates the denominator, so the softmax probability of the correct
                 positive drops ($0.79 \to 0.41$) and $-\log p$ rises. That is exactly the pressure NT-Xent puts
                 on the model: push hard (similar) negatives apart.

</details>

**Problem 3.** You build the $2N\times 2N$ similarity matrix and call cross-entropy, but training collapses — the
            loss instantly goes to near $0$ and the features are useless. You forgot one masking step. What is
            it, and why does omitting it collapse training?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether the diagonal (each view's similarity with itself) was masked out before the softmax. — _$\mathrm{sim}(z_i,z_i)=1$ for unit vectors — the largest possible — so the self term dominates the denominator and numerator comparison._
- Realize the model can "win" by matching each view to itself, which the indicator $\mathbb{1}_{[k\neq i]}$ is supposed to forbid. — _Without the mask the easiest solution is the trivial self-match, giving near-zero loss but learning nothing about the positive pair._
- Fix: set the diagonal of the similarity matrix to $-\infty$ (a large negative) before softmax. — _That implements $\mathbb{1}_{[k\neq i]}$ — removing the self term so the positive must be the other view._

**Answer:** You forgot to mask the diagonal (the self-similarity), i.e. the indicator
                 $\mathbb{1}_{[k\neq i]}$. Each view's similarity with itself is $1$ — the maximum — so the
                 model trivially "matches" every view to itself and drives the loss to ~$0$ while learning
                 nothing. Setting the diagonal to $-\infty$ before the softmax restores the real task: match each
                 view to its augmented partner.

</details>